In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("foodAllergyAnalysisZenodo.csv")

print(df.shape)
df.head()

(333200, 50)


,SUBJECT_ID,BIRTH_YEAR,GENDER_FACTOR,RACE_FACTOR,ETHNICITY_FACTOR,PAYER_FACTOR,ATOPIC_MARCH_COHORT,AGE_START_YEARS,AGE_END_YEARS,SHELLFISH_ALG_START,...,CASHEW_ALG_END,ATOPIC_DERM_START,ATOPIC_DERM_END,ALLERGIC_RHINITIS_START,ALLERGIC_RHINITIS_END,ASTHMA_START,ASTHMA_END,FIRST_ASTHMARX,LAST_ASTHMARX,NUM_ASTHMARX
0,1,2006,S1 - Female,R1 - Black,E0 - Non-Hispanic,P1 - Medicaid,False,0.093087,3.164956,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1994,S1 - Female,R0 - White,E0 - Non-Hispanic,P0 - Non-Medicaid,False,12.232717,18.880219,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.262834,18.880219,2.0
2,3,2006,S0 - Male,R0 - White,E1 - Hispanic,P0 - Non-Medicaid,True,0.010951,6.726899,NaN,...,NaN,4.884326,NaN,3.917864,6.157426,5.127995,NaN,1.404517,6.157426,4.0
3,4,2004,S0 - Male,R4 - Unknown,E1 - Hispanic,P0 - Non-Medicaid,False,2.398357,9.111567,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2006,S1 - Female,R1 - Black,E0 - Non-Hispanic,P0 - Non-Medicaid,False,0.013689,6.193018,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# If peanut allergy start age exists -> allergy = 1
# If NaN -> allergy = 0

df["PEANUT_ALLERGY"] = df["PEANUT_ALG_START"].notna().astype(int)
print(df["PEANUT_ALLERGY"].value_counts())

PEANUT_ALLERGY
0    324547
1      8653
Name: count, dtype: int64


In [17]:
# select useful features
features = [
    "AGE_START_YEARS",
    "GENDER_FACTOR",
    "RACE_FACTOR",
    "ETHNICITY_FACTOR",
    "ASTHMA_START",
    "ATOPIC_DERM_START"
]

target = "PEANUT_ALLERGY"
data = df[features + [target]].copy()

data = data.dropna()
print("Clean dataset shape:", data.shape)

Clean dataset shape: (15312, 7)


In [18]:
# convert categorical features
data = pd.get_dummies(data, drop_first=True)

In [19]:
# split the data
X = data.drop(target, axis=1)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# scale and train
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# predict probabilities
pred_probs = model.predict_proba(X_test)[:, 1]
pred_classes = model.predict(X_test)

# evaluate model
auc = roc_auc_score(y_test, pred_probs)

print("ROC-AUC Score:", auc)
print("\nClassification Report:\n")
print(classification_report(y_test, pred_classes))

ROC-AUC Score: 0.6132783834920503

Classification Report:

              precision    recall  f1-score   support

           0       0.88      1.00      0.94      2710
           1       0.33      0.00      0.01       353

    accuracy                           0.88      3063
   macro avg       0.61      0.50      0.47      3063
weighted avg       0.82      0.88      0.83      3063



In [20]:
# prediction function
def predict_peanut_allergy(age, gender, race, ethnicity, asthma, eczema):

    input_df = pd.DataFrame({
        "AGE_START_YEARS": [age],
        "GENDER_FACTOR": [gender],
        "RACE_FACTOR": [race],
        "ETHNICITY_FACTOR": [ethnicity],
        "ASTHMA_START": [asthma],
        "ATOPIC_DERM_START": [eczema]
    })

    input_df = pd.get_dummies(input_df)

    input_df = input_df.reindex(columns=X.columns, fill_value=0)

    input_scaled = scaler.transform(input_df)

    probability = model.predict_proba(input_scaled)[0][1]

    return probability

factors input:
gender: 1 → Male; 2 → Female 
race: 1 → White; 2 → Black; 3 → Asian; 4 → Other
ethnicity: 1 → Non-Hispanic; 2 → Hispanic
asthma: 0 → No asthma; 1 → Asthma present
eczema: 0 → No eczema; 1 → Eczema present

In [21]:
# example
risk = predict_peanut_allergy(
    age=5,
    gender=1,
    race=1,
    ethnicity=1,
    asthma=0,
    eczema=1
)

print("\nPredicted Peanut Allergy Risk:", risk)


Predicted Peanut Allergy Risk: 0.2824689099007222
